# 06 · Conformal Validation

Reads `runs/final_eval/conformal.json` (produced by `scripts/fit_conformal.py`) and reports:
- Per-coordinate quantiles `q̂ = (q_x1, q_y1, q_x2, q_y2)`.
- Empirical coverage on test vs claimed `1 - α` (should be ≥ target).
- CRC `λ*` and empirical FNR on test vs target `α`.

Regenerate: `python scripts/fit_conformal.py --calibration-split val --test-split test --alpha 0.05`.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
REPORT = ROOT / 'runs' / 'final_eval' / 'conformal.json'
assert REPORT.exists(), f'missing {REPORT}: run scripts/fit_conformal.py first'
report = json.loads(REPORT.read_text())
report

In [ ]:
sc = report['split_conformal']
crc = report['crc']
print(f'q_x1={sc["q"][0]:.4f}  q_y1={sc["q"][1]:.4f}  q_x2={sc["q"][2]:.4f}  q_y2={sc["q"][3]:.4f}')
print(f'coverage: empirical={sc["empirical_coverage_test"]:.4f}  target={sc["target_coverage"]:.4f}')
print(f'CRC λ*={crc["lambda_star"]:.4f}  FNR empirical={crc["empirical_fnr_test"]:.4f}  target ≤ {crc["target_fnr"]}')

In [ ]:
import numpy as np
# Sweep α if the report has additional records; else plot the point.
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.plot([0, 1], [0, 1], '--', color='0.6', label='target')
ax.scatter([sc['target_coverage']], [sc['empirical_coverage_test']],
           s=60, color='#1F5C4C', label='observed', zorder=3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
ax.set_xlabel('claimed coverage 1 − α'); ax.set_ylabel('empirical coverage (test)')
ax.set_title('Conformal box · empirical vs claimed')
ax.legend(loc='lower right', frameon=False)
plt.tight_layout(); plt.show()